In [12]:
# ==========================================
# BLOCO 1: CARREGAR DADOS
# ==========================================
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# Carregar dados processados
df_centroides = pd.read_csv('../data/processed/centroides_tribos.csv', index_col='Tribo_ID')
df_acoes = pd.read_csv('../data/processed/b3_vetor_financeiro.csv', index_col='Ticker')

In [13]:
# ==========================================
# BLOCO 2: CONSTRUÇÃO DO IPR (DOMAIN KNOWLEDGE)
# ==========================================
# A INTUIÇÃO: Abandonamos o PCA empilhado (que gera correlações espúrias).
# Em vez disso, a gente se baseia na análise de Loadings:
# PC1 mede Vulnerabilidade/Aperto Financeiro (logo, conservadorismo).
# PC2 mede Maturidade/Controle.
# Criamos uma heurística simples para a combinação dos dois e interpolando para a escala de 1 a 5.

# Criação do índice bruto (maior PC2 e menor PC1 indicam maior propensão teórica ao risco)
df_centroides['IPR_raw'] = df_centroides['PC2'] - df_centroides['PC1']

# Interpolação para a escala 1 (Conservador) a 5 (Agressivo)
df_centroides['IPR'] = np.interp(
    df_centroides['IPR_raw'],
    (df_centroides['IPR_raw'].min(), df_centroides['IPR_raw'].max()),
    (1, 5)
)

print("=== ÍNDICE DE PERFIL DE RISCO (IPR) DAS TRIBOS ===")
print(df_centroides[['IPR_raw', 'IPR']].sort_values('IPR'))

=== ÍNDICE DE PERFIL DE RISCO (IPR) DAS TRIBOS ===
           IPR_raw       IPR
Tribo_ID                    
4        -3.312637  1.000000
3        -2.959873  1.208223
1        -2.300436  1.597463
6         0.363952  3.170147
2         0.401022  3.192028
0         1.038799  3.568482
5         3.464027  5.000000


In [14]:
# ==========================================
# BLOCO 3: MATCHMAKING E RECOMENDAÇÃO
# ==========================================
# Extraímos as matrizes unidimensionais (IPR vs IRA)
matriz_ipr = df_centroides[['IPR']].values
matriz_ira = df_acoes[['IRA']].values

# Cálculo de similaridade usando a distância Euclidiana absoluta
distancias = cdist(matriz_ipr, matriz_ira, metric='euclidean')

df_distancias = pd.DataFrame(
    distancias,
    index=df_centroides.index,
    columns=df_acoes.index
)

recomendacoes = {}
for tribo in df_distancias.index:
    # Pega os 5 Tickers com a MENOR distância (maior similaridade)
    top_acoes = df_distancias.loc[tribo].nsmallest(5)
    recomendacoes[tribo] = top_acoes.index.tolist()

df_recomendacoes = pd.DataFrame(recomendacoes).T
df_recomendacoes.columns = [f'Top_{i+1}' for i in range(5)]
df_recomendacoes.to_csv('../data/processed/recomendacoes_finais_domain_knowledge.csv')

print("\n=== RECOMENDAÇÕES (RISK-BUCKET MATCHING) ===")
print(df_recomendacoes)
print("\nRecomendações salvas com sucesso!")


=== RECOMENDAÇÕES (RISK-BUCKET MATCHING) ===
      Top_1     Top_2     Top_3     Top_4     Top_5
0  BBDC4.SA  LREN3.SA  QUAL3.SA  DXCO3.SA  RENT3.SA
1  KLBN4.SA  BAZA3.SA  VIVT3.SA  EGIE3.SA  ABEV3.SA
2  ITUB4.SA  SANB4.SA  BRSR6.SA  CSAN3.SA  ITSA4.SA
3  SUZB3.SA  KLBN4.SA  BAZA3.SA  VIVT3.SA  EGIE3.SA
4  SUZB3.SA  KLBN4.SA  BAZA3.SA  VIVT3.SA  EGIE3.SA
5  CSNA3.SA  MGLU3.SA  USIM5.SA  PRIO3.SA  PETR4.SA
6  ITUB4.SA  SANB4.SA  ITSA4.SA  BRSR6.SA  CSAN3.SA

Recomendações salvas com sucesso!


In [15]:
# ==========================================
# BLOCO 4: VALIDAÇÃO DE NEGÓCIO
# ==========================================
# Verifica se o perfil mais agressivo recebeu ações mais arriscadas
# do que o perfil mais conservador.

perfil_conservador = df_centroides['IPR'].idxmin()
perfil_agressivo = df_centroides['IPR'].idxmax()

acoes_cons = df_recomendacoes.loc[perfil_conservador].tolist()
acoes_agr = df_recomendacoes.loc[perfil_agressivo].tolist()

media_beta_cons = df_acoes.loc[acoes_cons, 'Beta'].mean()
media_beta_agr = df_acoes.loc[acoes_agr, 'Beta'].mean()

print("\n=== Validação de Negócio ===")
print(f"Perfil Mais Conservador (ID {perfil_conservador}) - IPR: {df_centroides.loc[perfil_conservador, 'IPR']:.2f}")
print(f" -> Média Beta das Recomendadas: {media_beta_cons:.2f}")

print(f"\nPerfil Mais Agressivo (ID {perfil_agressivo}) - IPR: {df_centroides.loc[perfil_agressivo, 'IPR']:.2f}")
print(f" -> Média Beta das Recomendadas: {media_beta_agr:.2f}")

if media_beta_agr > media_beta_cons:
    print("\n[SUCESSO] O modelo está logicamente consistente: Pefis agressivos recebem ativos de maior Beta.")
else:
    print("\n[ALERTA] Inconsistência detectada. Verifique os cálculos do IPR/IRA.")


=== SANITY CHECK ===
Tribo Mais Conservadora (ID 4) - IPR: 1.00
 -> Média Beta das Recomendadas: 0.41

Tribo Mais Agressiva (ID 5) - IPR: 5.00
 -> Média Beta das Recomendadas: 1.29

[SUCESSO] O modelo está logicamente consistente: Tribos agressivas recebem ativos de maior Beta.
